# EuroCup API — Explorare date U-BT

Notebook de explorare a datelor disponibile prin `euroleague-api`.  
Rulează celulele în ordine.

**Competiție:** `'U'` = EuroCup &nbsp;|&nbsp; `'E'` = Euroleague  
**Season:** anul de start (ex: `2025` = sezonul 2025-26)  
**TEAM_CODE:** `'CLU'` = U-BT Cluj-Napoca (confirmat din API)

---
### Descoperiri cheie
- `get_game_metadata_single_season()` returnează coloane **PascalCase**: `CodeTeamA`, `TeamA`, `ScoreA`, `Stadium`, `Capacity` etc.
- `get_player_stats_single_season(endpoint='traditional', statistic_mode='Accumulated')` returnează **toți 14 jucători** — `PerGame` exclude jucătorii cu minute puține
- `Capacity` = spectatori prezenți (nu capacitatea arenei)
- Sezonul regulat 2025-26: **20 meciuri** complete

## 0. Setup

In [ ]:
import sys
from pathlib import Path

import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 40)
pd.set_option('display.float_format', '{:.2f}'.format)

# Add project root to path
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

from euroleague_api.game_metadata import GameMetadata
from euroleague_api.player_stats  import PlayerStats
from euroleague_api.team_stats    import TeamStats
from euroleague_api.standings     import Standings
from euroleague_api.game_stats    import GameStats

COMPETITION = 'U'     # EuroCup
SEASON      = 2025    # 2025-26
TEAM_CODE   = 'CLU'   # U-BT Cluj-Napoca — confirmat din API

print('Setup OK')

## 1. Game Codes — meciuri din sezon

In [ ]:
gm = GameMetadata(COMPETITION)

df_codes = gm.get_gamecodes_season(season=SEASON)
print(f'Total meciuri în sezon: {len(df_codes)}')
print(f'Coloane: {list(df_codes.columns)}')
df_codes.head(10)

## 2. Game Metadata — toate meciurile + filtrare U-BT

⚠️ Coloane PascalCase: `CodeTeamA/B`, `TeamA/B`, `ScoreA/B`, `Stadium`, `Capacity`, `Phase`

In [ ]:
df_games_all = gm.get_game_metadata_single_season(season=SEASON)
print(f'Total meciuri: {len(df_games_all)}')
print(f'Coloane ({len(df_games_all.columns)}): {list(df_games_all.columns)}')
df_games_all.head(3)

In [ ]:
# Filtrare U-BT
mask = (df_games_all['CodeTeamA'] == TEAM_CODE) | (df_games_all['CodeTeamB'] == TEAM_CODE)
df_ubt = df_games_all[mask].copy()

print(f'Meciuri U-BT: {len(df_ubt)}')
df_ubt[['Round','Date','TeamA','TeamB','ScoreA','ScoreB','Stadium','Capacity']]

## 3. Normalizare Games

In [ ]:
df_games_norm = pd.DataFrame({
    'competition':  'EuroCup',
    'season':       SEASON,
    'game_code':    df_ubt['Gamecode'],
    'round':        df_ubt['Round'],
    'date':         pd.to_datetime(df_ubt['Date'], format='%d/%m/%Y'),
    'home_team':    df_ubt['TeamA'],
    'home_code':    df_ubt['CodeTeamA'],
    'away_team':    df_ubt['TeamB'],
    'away_code':    df_ubt['CodeTeamB'],
    'score_home':   pd.to_numeric(df_ubt['ScoreA'], errors='coerce'),
    'score_away':   pd.to_numeric(df_ubt['ScoreB'], errors='coerce'),
    'venue':        df_ubt['Stadium'],
    'attendance':   pd.to_numeric(df_ubt['Capacity'], errors='coerce'),
    'phase':        df_ubt['Phase'],
})

# Derived
df_games_norm['ubt_is_home'] = df_games_norm['home_code'] == TEAM_CODE
df_games_norm['ubt_score']   = df_games_norm.apply(lambda r: r['score_home'] if r['ubt_is_home'] else r['score_away'], axis=1)
df_games_norm['opp_score']   = df_games_norm.apply(lambda r: r['score_away'] if r['ubt_is_home'] else r['score_home'], axis=1)
df_games_norm['opponent']    = df_games_norm.apply(lambda r: r['away_team'] if r['ubt_is_home'] else r['home_team'], axis=1)
df_games_norm['result']      = df_games_norm.apply(lambda r: 'W' if r['ubt_score'] > r['opp_score'] else 'L', axis=1)
df_games_norm['score_diff']  = df_games_norm['ubt_score'] - df_games_norm['opp_score']
df_games_norm.reset_index(drop=True, inplace=True)

print(f'Games normalizate: {len(df_games_norm)}')
df_games_norm[['date','opponent','ubt_score','opp_score','result','score_diff','venue','attendance']]

## 4. Player Stats

⚠️ Folosim `Accumulated` — `PerGame` exclude jucătorii cu minute puține (Miron, Petric, Mensah)

In [ ]:
ps = PlayerStats(COMPETITION)

# Accumulated = totaluri pe sezon, toți jucătorii
df_ps_acc = ps.get_player_stats_single_season(
    endpoint='traditional',
    season=SEASON,
    statistic_mode='Accumulated'
)

# Filtrare U-BT
df_ubt_acc = df_ps_acc[df_ps_acc['player.team.code'] == TEAM_CODE].copy()
print(f'Jucători U-BT: {len(df_ubt_acc)}')
df_ubt_acc[['player.name','gamesPlayed','minutesPlayed','pointsScored']].sort_values('minutesPlayed', ascending=False)

## 5. Normalizare Player Stats

In [ ]:
df_players_norm = pd.DataFrame({
    'competition':  'EuroCup',
    'season':       SEASON,
    'player_id':    df_ubt_acc['player.code'],
    'player_name':  df_ubt_acc['player.name'],
    'team_code':    TEAM_CODE,
    'games_played': df_ubt_acc['gamesPlayed'],
    'minutes':      df_ubt_acc['minutesPlayed'],
    'points':       df_ubt_acc['pointsScored'],
    'rebounds':     df_ubt_acc['totalRebounds'],
    'off_rebounds': df_ubt_acc['offensiveRebounds'],
    'def_rebounds': df_ubt_acc['defensiveRebounds'],
    'assists':      df_ubt_acc['assists'],
    'steals':       df_ubt_acc['steals'],
    'blocks':       df_ubt_acc['blocks'],
    'turnovers':    df_ubt_acc['turnovers'],
    'fouls':        df_ubt_acc['foulsCommited'],
    'fg2_made':     df_ubt_acc['twoPointersMade'],
    'fg2_att':      df_ubt_acc['twoPointersAttempted'],
    'fg3_made':     df_ubt_acc['threePointersMade'],
    'fg3_att':      df_ubt_acc['threePointersAttempted'],
    'ft_made':      df_ubt_acc['freeThrowsMade'],
    'ft_att':       df_ubt_acc['freeThrowsAttempted'],
    'pir':          df_ubt_acc['pir'],
    'image_url':    df_ubt_acc['player.imageUrl'],
})

# Medii per game
for col in ['points','rebounds','assists','steals','blocks','turnovers','minutes']:
    df_players_norm[f'{col}_pg'] = (df_players_norm[col] / df_players_norm['games_played']).round(2)

# Procentaje
df_players_norm['fg2_pct'] = (df_players_norm['fg2_made'] / df_players_norm['fg2_att']).round(3)
df_players_norm['fg3_pct'] = (df_players_norm['fg3_made'] / df_players_norm['fg3_att']).round(3)
df_players_norm['ft_pct']  = (df_players_norm['ft_made']  / df_players_norm['ft_att']).round(3)

df_players_norm.reset_index(drop=True, inplace=True)
print(f'Jucători normalizați: {len(df_players_norm)}')
df_players_norm[['player_name','games_played','minutes_pg','points_pg','rebounds_pg','assists_pg','pir']]

## 6. Export CSV

In [ ]:
processed_dir = ROOT / 'data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)

games_path   = processed_dir / 'games.csv'
players_path = processed_dir / 'player_stats.csv'

df_games_norm.to_csv(games_path, index=False)
df_players_norm.to_csv(players_path, index=False)

print(f'✓ {games_path}  ({len(df_games_norm)} rânduri)')
print(f'✓ {players_path}  ({len(df_players_norm)} rânduri)')

## 7. Extra — Team Stats & Standings (opțional)

In [ ]:
# Team stats U-BT
ts = TeamStats(COMPETITION)
df_team = ts.get_team_stats_single_season(endpoint='teams', season=SEASON, statistic_mode='Accumulated')
df_ubt_team = df_team[df_team['team.code'] == TEAM_CODE] if 'team.code' in df_team.columns else df_team
print(f'Coloane team stats: {list(df_team.columns)}')
df_ubt_team

In [ ]:
# Standings — încearcă ultima rundă disponibilă
s = Standings(COMPETITION)
for rnd in [20, 19, 18, 10, 5, 1]:
    try:
        df_st = s.get_standings(season=SEASON, round_number=rnd)
        print(f'Standings round {rnd}: {len(df_st)} echipe')
        print(df_st[['cname','played','wins','losses']].to_string())
        break
    except Exception as e:
        print(f'Round {rnd}: {e}')